# 01 · Data check: Electric Motor Temperature dataset

**Goal:** Load the raw data, check that it is what we expect (size, sessions, missing values, duplicates, physical ranges), and save a smaller, faster copy for the next notebooks.

**Inputs:**
- `data/raw/measures_v2.csv`: Paderborn University PMSM test-bench data from Kaggle (`wkirgsn/electric-motor-temperature`, version 3). Download it with `src/download_data.py`.

**Outputs:**
- `data/processed/measures_v2.parquet`: the same data, with float columns stored as `float32`.
- `outputs/results/data_check_summary.json`: every number this notebook finds.
- `outputs/results/rows_per_profile.csv`: the length of each recording session.

**Phase:** 0 (Setup & data check). Nothing here is a model. We only *look* at the data and make sure it is sane.

## 1. Setup
Import the libraries, fix the random seed (`SEED = 42`, our team rule for reproducibility) and define all paths with `pathlib`.
Our folder path has spaces and brackets, so we never type paths as plain strings.

In [1]:
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# This notebook lives in notebooks/, so the project folder is one level up
if Path.cwd().name == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

RAW_CSV = PROJECT_DIR / "data" / "raw" / "measures_v2.csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE_HZ = 2  # from the dataset description: one row every 0.5 s

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("Raw CSV found:", RAW_CSV.exists())

Python : 3.11.14
pandas : 3.0.6
numpy  : 2.4.6
Raw CSV found: True


## 2. Load the CSV
We read the whole file into a pandas DataFrame (`df`) and time how long it takes.
Later we compare this with the parquet file to show why we switch formats.

In [2]:
start = time.time()
df = pd.read_csv(RAW_CSV)
csv_load_seconds = time.time() - start

csv_size_mb = RAW_CSV.stat().st_size / 1e6
print(f"CSV size on disk : {csv_size_mb:.1f} MB")
print(f"Load time        : {csv_load_seconds:.1f} s")
print("Shape (rows, cols):", df.shape)

CSV size on disk : 300.1 MB
Load time        : 7.7 s
Shape (rows, cols): (1330816, 13)


### 2.1 Column types
All 12 measurement columns should be decimal numbers (`float64`). `profile_id` should be a whole number (`int64`), because it is the ID of a recording session.

In [3]:
df.dtypes

u_q               float64
coolant           float64
stator_winding    float64
u_d               float64
stator_tooth      float64
motor_speed       float64
i_d               float64
i_q               float64
pm                float64
stator_yoke       float64
ambient           float64
torque            float64
profile_id          int64
dtype: object

### 2.2 First rows
A first look at the values. Units (from the dataset description): voltages in V, currents in A, `motor_speed` in rpm, `torque` in Nm, all temperatures in °C.

In [4]:
df.head()

,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,0.187101,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,0.245417,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,0.176615,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,0.238303,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,0.208197,17


### 2.3 Memory used in RAM
`deep=True` makes pandas count the real memory of every column.

In [5]:
mem_float64_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Memory used by df: {mem_float64_mb:.1f} MB")

Memory used by df: 138.4 MB


## 3. Recording sessions (`profile_id`)
Each `profile_id` is one measurement session on the test bench. Sessions matter a lot for us:
- we will split train and test **by whole sessions** (plan nuance N2);
- sessions have very different lengths (nuance N5).

We count the sessions and convert each one's row count into hours: `hours = rows / 2 Hz / 3600 s`.

In [6]:
rows_per_profile = df.groupby("profile_id").size()
hours_per_profile = rows_per_profile / SAMPLE_RATE_HZ / 3600

n_profiles = df["profile_id"].nunique()
total_hours = len(df) / SAMPLE_RATE_HZ / 3600
print("Number of profiles (sessions):", n_profiles)
print(f"Total recording time        : {total_hours:.1f} hours")
print("Smallest / largest profile_id:", rows_per_profile.index.min(), "/", rows_per_profile.index.max())

profile_summary = pd.DataFrame({
    "rows": [rows_per_profile.min(), rows_per_profile.median(), rows_per_profile.max()],
    "hours": [hours_per_profile.min(), hours_per_profile.median(), hours_per_profile.max()],
}, index=["min", "median", "max"])
profile_summary.round(2)

Number of profiles (sessions): 69
Total recording time        : 184.8 hours
Smallest / largest profile_id: 2 / 81


,rows,hours
min,2176.0,0.30
median,17142.0,2.38
max,43971.0,6.11


Save the length of every session to a CSV. Phase 3 needs it to pick a test set that has both short and long sessions.

In [7]:
profile_table = pd.DataFrame({
    "profile_id": rows_per_profile.index,
    "n_rows": rows_per_profile.values,
    "hours": hours_per_profile.values.round(3),
})
profile_table = profile_table.sort_values("n_rows").reset_index(drop=True)
profile_table.to_csv(RESULTS_DIR / "rows_per_profile.csv", index=False)

print("3 shortest sessions:")
print(profile_table.head(3).to_string(index=False))
print("\n3 longest sessions:")
print(profile_table.tail(3).to_string(index=False))

3 shortest sessions:
 profile_id  n_rows  hours
         47    2176  0.302
         46    2180  0.303
         52    3726  0.518

3 longest sessions:
 profile_id  n_rows  hours
         65   40094  5.569
          6   40388  5.609
         20   43971  6.107


## 4. Is the file in time order? (plan nuance N9)
The file has **no timestamp column**. Our EWMA "thermal memory" features (Phase 3) only make sense if the rows inside each session are in time order. We can't prove that directly, but we can test two things that *must* be true if it holds:

1. **Each session is one unbroken block of rows.** If a `profile_id` showed up in several separate places, the order would be suspicious.
2. **Temperature changes very little from one row to the next.** A motor's winding has thermal inertia, so in 0.5 s its temperature can barely move. If the rows were shuffled, the jumps between neighbouring rows would be as big as the jumps between random rows.

In [8]:
# A new block starts wherever profile_id differs from the row above
block_starts = df["profile_id"] != df["profile_id"].shift()
profile_order_in_file = df.loc[block_starts, "profile_id"].tolist()
blocks_per_profile = pd.Series(profile_order_in_file).value_counts()
n_profiles_not_contiguous = int((blocks_per_profile > 1).sum())

print("Number of blocks in the file :", len(profile_order_in_file))
print("Profiles split into >1 block :", n_profiles_not_contiguous)
print("File sorted by profile_id?   :", profile_order_in_file == sorted(profile_order_in_file))
print("First 10 profile_ids in file order:", profile_order_in_file[:10])

Number of blocks in the file : 69
Profiles split into >1 block : 0
File sorted by profile_id?   : False
First 10 profile_ids in file order: [17, 5, 12, 32, 21, 2, 4, 7, 19, 3]


In [9]:
# Change in winding temperature between neighbouring rows, inside each session
step_file_order = df.groupby("profile_id")["stator_winding"].diff().abs()

# The same, after shuffling all rows (what we'd see if row order meant nothing)
shuffled = df["stator_winding"].sample(frac=1, random_state=SEED).reset_index(drop=True)
step_shuffled = shuffled.diff().abs()

median_step_file = step_file_order.median()
median_step_shuffled = step_shuffled.median()
p999_step_file = step_file_order.quantile(0.999)
max_step_file = step_file_order.max()

print(f"Median row-to-row change, file order    : {median_step_file:.4f} °C")
print(f"Median row-to-row change, shuffled order: {median_step_shuffled:.4f} °C")
print(f"99.9th percentile change, file order    : {p999_step_file:.4f} °C")
print(f"Largest change, file order              : {max_step_file:.4f} °C")

Median row-to-row change, file order    : 0.0221 °C
Median row-to-row change, shuffled order: 28.8548 °C
99.9th percentile change, file order    : 0.5549 °C
Largest change, file order              : 1.3939 °C


## 5. Missing values and duplicate rows
- **Missing values (NaN)** would need filling or dropping in Phase 3.
- **Exact duplicate rows** could mean a logging error. We check two versions: all 13 columns, and the 12 measurements without `profile_id` (the same reading appearing in two different sessions).

In [10]:
missing_per_column = df.isna().sum()
n_missing_total = int(missing_per_column.sum())
n_duplicate_rows = int(df.duplicated().sum())
n_duplicate_rows_no_id = int(df.drop(columns="profile_id").duplicated().sum())

print("Missing values per column:")
print(missing_per_column.to_string())
print("\nTotal missing values                    :", n_missing_total)
print("Exact duplicate rows (all 13 columns)   :", n_duplicate_rows)
print("Duplicate rows ignoring profile_id      :", n_duplicate_rows_no_id)

Missing values per column:
u_q               0
coolant           0
stator_winding    0
u_d               0
stator_tooth      0
motor_speed       0
i_d               0
i_q               0
pm                0
stator_yoke       0
ambient           0
torque            0
profile_id        0

Total missing values                    : 0
Exact duplicate rows (all 13 columns)   : 0
Duplicate rows ignoring profile_id      : 0


## 6. Value ranges: are the numbers physically sensible?
`describe()` gives the count, mean, standard deviation (std), min, quartiles and max of every column.

In [11]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
u_q,1330816.0,54.279,44.173,-25.291,12.070,48.938,90.034,133.037
coolant,1330816.0,36.230,21.786,10.624,18.698,26.900,49.857,101.599
stator_winding,1330816.0,66.343,28.672,18.586,42.788,65.110,88.141,141.363
u_d,1330816.0,-25.134,63.092,-131.530,-78.691,-7.430,1.470,131.470
stator_tooth,1330816.0,56.879,22.952,18.134,38.416,56.036,75.587,111.946
motor_speed,1330816.0,2202.081,1859.663,-275.549,317.111,1999.977,3760.639,6000.015
i_d,1330816.0,-68.717,64.933,-278.004,-115.406,-51.094,-2.980,0.052
i_q,1330816.0,37.413,92.182,-293.427,1.096,15.774,100.612,301.708
pm,1330816.0,58.507,19.001,20.857,43.152,60.266,72.008,113.607
stator_yoke,1330816.0,48.188,19.991,18.077,31.990,45.626,61.461,101.148


### 6.1 Temperature range check
We check every temperature column against plausible limits for a motor on a test bench. We chose **-20 °C to 200 °C**: below -20 °C the lab would be frozen, and 200 °C is above the strongest common insulation class (Class H = 180 °C).
A value outside this range would most likely be a sensor error.

In [12]:
temp_cols = ["coolant", "ambient", "stator_winding", "stator_tooth", "stator_yoke", "pm"]
TEMP_MIN_C = -20
TEMP_MAX_C = 200

range_rows = []
for col in temp_cols:
    col_min = float(df[col].min())
    col_max = float(df[col].max())
    inside = (col_min >= TEMP_MIN_C) and (col_max <= TEMP_MAX_C)
    range_rows.append({"column": col, "min_C": round(col_min, 2), "max_C": round(col_max, 2), "inside_limits": inside})

range_check = pd.DataFrame(range_rows)
range_check

,column,min_C,max_C,inside_limits
0,coolant,10.62,101.60,True
1,ambient,8.78,30.71,True
2,stator_winding,18.59,141.36,True
3,stator_tooth,18.13,111.95,True
4,stator_yoke,18.08,101.15,True
5,pm,20.86,113.61,True


### 6.2 Speed and torque signs
Negative speed means the rotor turns the other way. Negative torque means the motor is **braking** (working as a generator). Both are real operating points, not errors, so we only count them.

In [13]:
n_negative_speed = int((df["motor_speed"] < 0).sum())
n_negative_torque = int((df["torque"] < 0).sum())

print(f"motor_speed range: {df['motor_speed'].min():.1f} to {df['motor_speed'].max():.1f} rpm")
print(f"torque range     : {df['torque'].min():.1f} to {df['torque'].max():.1f} Nm")
print(f"Rows with negative speed : {n_negative_speed} ({100 * n_negative_speed / len(df):.2f} %)")
print(f"Rows with negative torque: {n_negative_torque} ({100 * n_negative_torque / len(df):.2f} %)")

motor_speed range: -275.5 to 6000.0 rpm
torque range     : -246.5 to 261.0 Nm
Rows with negative speed : 87308 (6.56 %)
Rows with negative torque: 446946 (33.58 %)


## 7. Save a smaller copy as parquet (float32)
- **float64 → float32:** halves the memory of every decimal column. float32 keeps about 7 significant digits, far more than a temperature sensor can measure. We check how much any value actually changed.
- **CSV → parquet:** parquet is a compressed, column-based binary format that remembers dtypes. It is much smaller and faster to load than text CSV.

In [14]:
df_32 = df.copy()
float_cols = df_32.select_dtypes(include="float64").columns.tolist()
df_32[float_cols] = df_32[float_cols].astype("float32")

mem_float32_mb = df_32.memory_usage(deep=True).sum() / 1e6
memory_saved_pct = 100 * (1 - mem_float32_mb / mem_float64_mb)

# Biggest change in a single value caused by the cast, for each column
change_per_column = (df_32[float_cols].astype("float64") - df[float_cols]).abs().max()
max_value_change = float(change_per_column.max())

print("Columns cast to float32:", len(float_cols))
print(f"Memory float64: {mem_float64_mb:.1f} MB -> float32: {mem_float32_mb:.1f} MB (saved {memory_saved_pct:.1f} %)")
print("\nLargest change in any value after the cast, per column (in that column's unit):")
print(change_per_column.sort_values(ascending=False).to_string())

Columns cast to float32: 12
Memory float64: 138.4 MB -> float32: 74.5 MB (saved 46.2 %)

Largest change in any value after the cast, per column (in that column's unit):
motor_speed       2.441402e-04
i_q               1.525826e-05
i_d               1.466250e-05
torque            7.629391e-06
u_q               7.629331e-06
u_d               7.629282e-06
stator_winding    7.621336e-06
coolant           3.814695e-06
pm                3.814693e-06
stator_yoke       3.814686e-06
stator_tooth      3.814663e-06
ambient           9.536735e-07


In [15]:
PARQUET_PATH = PROCESSED_DIR / "measures_v2.parquet"
df_32.to_parquet(PARQUET_PATH, index=False)
parquet_size_mb = PARQUET_PATH.stat().st_size / 1e6

# Reload to check the file is correct and to time it
start = time.time()
df_check = pd.read_parquet(PARQUET_PATH)
parquet_load_seconds = time.time() - start

same_shape = df_check.shape == df.shape
same_row_order = bool((df_check["profile_id"].to_numpy() == df["profile_id"].to_numpy()).all())

print(f"Parquet size on disk: {parquet_size_mb:.1f} MB (CSV was {csv_size_mb:.1f} MB)")
print(f"Load time: parquet {parquet_load_seconds:.2f} s vs CSV {csv_load_seconds:.1f} s")
print("Same shape after reload    :", same_shape)
print("Same row order after reload:", same_row_order)

Parquet size on disk: 72.2 MB (CSV was 300.1 MB)
Load time: parquet 0.42 s vs CSV 7.7 s
Same shape after reload    : True
Same row order after reload: True


## 8. Save every number to a results file
Team rule: every number we quote in the log, slides or report must come from a results file like this one.

In [16]:
summary = {
    "csv_file": "data/raw/measures_v2.csv",
    "kaggle_dataset": "wkirgsn/electric-motor-temperature (version 3)",
    "csv_size_mb": round(csv_size_mb, 1),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "columns": df.columns.tolist(),
    "n_profiles": int(n_profiles),
    "profile_id_min": int(rows_per_profile.index.min()),
    "profile_id_max": int(rows_per_profile.index.max()),
    "total_hours": round(float(total_hours), 2),
    "rows_per_profile_min": int(rows_per_profile.min()),
    "rows_per_profile_median": float(rows_per_profile.median()),
    "rows_per_profile_max": int(rows_per_profile.max()),
    "hours_per_profile_min": round(float(hours_per_profile.min()), 3),
    "hours_per_profile_median": round(float(hours_per_profile.median()), 3),
    "hours_per_profile_max": round(float(hours_per_profile.max()), 3),
    "n_blocks_in_file": len(profile_order_in_file),
    "n_profiles_not_contiguous": n_profiles_not_contiguous,
    "file_sorted_by_profile_id": profile_order_in_file == sorted(profile_order_in_file),
    "median_step_stator_winding_file_order_C": round(float(median_step_file), 5),
    "median_step_stator_winding_shuffled_C": round(float(median_step_shuffled), 5),
    "p999_step_stator_winding_file_order_C": round(float(p999_step_file), 5),
    "max_step_stator_winding_file_order_C": round(float(max_step_file), 5),
    "missing_values_total": n_missing_total,
    "duplicate_rows": n_duplicate_rows,
    "duplicate_rows_ignoring_profile_id": n_duplicate_rows_no_id,
    "temperature_limits_C": [TEMP_MIN_C, TEMP_MAX_C],
    "temperature_range_check": [
        {"column": r["column"], "min_C": float(r["min_C"]), "max_C": float(r["max_C"]), "inside_limits": bool(r["inside_limits"])}
        for r in range_rows
    ],
    "motor_speed_min_rpm": round(float(df["motor_speed"].min()), 2),
    "motor_speed_max_rpm": round(float(df["motor_speed"].max()), 2),
    "torque_min_Nm": round(float(df["torque"].min()), 2),
    "torque_max_Nm": round(float(df["torque"].max()), 2),
    "negative_speed_rows": n_negative_speed,
    "negative_torque_rows": n_negative_torque,
    "memory_float64_mb": round(float(mem_float64_mb), 1),
    "memory_float32_mb": round(float(mem_float32_mb), 1),
    "memory_saved_pct": round(float(memory_saved_pct), 1),
    "max_value_change_after_float32": max_value_change,
    "max_value_change_after_float32_per_column": {col: float(v) for col, v in change_per_column.items()},
    "parquet_size_mb": round(parquet_size_mb, 1),
    "csv_load_seconds": round(csv_load_seconds, 2),
    "parquet_load_seconds": round(parquet_load_seconds, 2),
}

with open(RESULTS_DIR / "data_check_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", RESULTS_DIR / "data_check_summary.json")

Saved: D:\TIET Stuff\Semester 3\AI for Engineers (UCS321)\Project stuff\MST_Motor_Temperature\outputs\results\data_check_summary.json


## 9. What we found
- **Size:** 1,330,816 rows × 13 columns, in **69 sessions**, **184.8 hours** in total. This matches the dataset description (about 1.3M rows, 69 profiles, about 185 h).
- **Sessions are very unequal:** from 0.30 h (profile 47) to 6.11 h (profile 20), median 2.38 h. We must keep this in mind when we pick test sessions (nuance N5).
- **IDs have gaps:** `profile_id` runs from 2 to 81, but only 69 IDs exist. Never assume the IDs are 1, 2, 3, …
- **Row order:** every session is one unbroken block of rows, but the blocks are *not* sorted by ID (the file starts 17, 5, 12, …). The median row-to-row change in winding temperature is **0.022 °C** in file order vs **28.85 °C** after shuffling. So the rows inside each session really are in time order, and our N9 assumption holds. **Never sort or shuffle before the time features are built.**
- **Clean data:** 0 missing values and 0 duplicate rows. No cleaning is needed for these.
- **Ranges are physical:** every temperature is inside −20…200 °C. The hottest winding reading is **141.36 °C**, below the Class F limit (155 °C).
- **Signs:** 6.56 % of rows have negative speed (down to −275.5 rpm) and 33.58 % have negative torque. The 25th percentile of torque is only −0.137 Nm, so many "negative" rows are tiny values around zero at standstill, not hard braking.
- **`i_d` is almost never positive** (max 0.052 A). This fits PMSM control: a negative d-current weakens the magnet's field at high speed (field weakening).
- **float32 + parquet:** memory 138.4 → 74.5 MB (46.2 % saved). The biggest change in any value is 0.000244 rpm (`motor_speed`), and the target changed by at most 0.0000076 °C. The parquet file is 72.2 MB vs 300.1 MB for the CSV, and loads several times faster (the exact times are in section 7's output and the JSON; they vary a little from run to run).